# Variational Autoencoder (VAE) pada MNIST

**Buku:** *Deep Learning dengan Python* — Bab 14, Deep Learning Generatif

Encoder memetakan gambar ke distribusi di ruang laten 2 dimensi; decoder membangkitkan angka baru dari titik mana pun di ruang itu.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Notebook ini **membutuhkan backend TensorFlow** karena memakai `tf.GradientTape` untuk gradien terhadap masukan.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- **Bug asli diperbaiki:** sampling memakai `exp(z_log_var)` sebagai simpangan baku. Karena `z_log_var` adalah log-**varians**, yang benar adalah `exp(0.5 * z_log_var)`.
- Lapisan loss kustom dengan `add_loss` dan `compile(loss=None)` → subclass `keras.Model` dengan `train_step`, dengan loss rekonstruksi dan KL dilacak terpisah.
- Loss rekonstruksi dijumlahkan per piksel (bukan dirata-ratakan) agar seimbang dengan suku KL.
- `K.random_normal` → `keras.random.normal` dengan `SeedGenerator`.

In [ ]:
import os
# Backend wajib TensorFlow untuk notebook ini
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf
import numpy as np
from keras import layers, ops

## 1. Lapisan sampling (trik reparameterisasi)
$z = \mu + \sigma\,\varepsilon$ dengan $\sigma = \exp(\tfrac12 \log\sigma^2)$ dan $\varepsilon \sim \mathcal N(0, I)$.

In [ ]:
class Sampling(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.generator_acak = keras.random.SeedGenerator(1337)

    def call(self, masukan):
        z_mean, z_log_var = masukan
        eps = keras.random.normal(shape=ops.shape(z_mean), seed=self.generator_acak)
        return z_mean + ops.exp(0.5 * z_log_var) * eps    # asli: exp(z_log_var) (salah)

## 2. Encoder dan decoder

In [ ]:
DIM_LATEN = 2

masukan_enc = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(masukan_enc)
x = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation="relu")(x)
z_mean = layers.Dense(DIM_LATEN, name="z_mean")(x)
z_log_var = layers.Dense(DIM_LATEN, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(masukan_enc, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

masukan_dec = keras.Input(shape=(DIM_LATEN,))
x = layers.Dense(7 * 7 * 64, activation="relu")(masukan_dec)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
keluaran_dec = layers.Conv2D(1, 3, activation="sigmoid", padding="same")(x)
decoder = keras.Model(masukan_dec, keluaran_dec, name="decoder")
decoder.summary()

## 3. Model VAE dengan `train_step` kustom

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder, self.decoder = encoder, decoder
        self.pelacak_total = keras.metrics.Mean(name="loss_total")
        self.pelacak_rekon = keras.metrics.Mean(name="loss_rekonstruksi")
        self.pelacak_kl = keras.metrics.Mean(name="loss_kl")

    @property
    def metrics(self):
        return [self.pelacak_total, self.pelacak_rekon, self.pelacak_kl]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            rekonstruksi = self.decoder(z)
            loss_rekon = ops.mean(ops.sum(
                keras.losses.binary_crossentropy(data, rekonstruksi), axis=(1, 2)))
            kl = -0.5 * (1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var))
            loss_kl = ops.mean(ops.sum(kl, axis=1))
            loss_total = loss_rekon + loss_kl
        grad = tape.gradient(loss_total, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grad, self.trainable_weights))
        self.pelacak_total.update_state(loss_total)
        self.pelacak_rekon.update_state(loss_rekon)
        self.pelacak_kl.update_state(loss_kl)
        return {m.name: m.result() for m in self.metrics}

## 4. Latih (gambar latih + uji, tanpa label)

In [ ]:
(x_latih, _), (x_uji, _) = keras.datasets.mnist.load_data()
angka = np.concatenate([x_latih, x_uji], axis=0)
angka = np.expand_dims(angka, -1).astype("float32") / 255

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())
vae.fit(angka, epochs=30, batch_size=128)

## 5. Menjelajahi ruang laten
Kisi titik diambil dari kuantil distribusi normal sehingga kepadatannya mengikuti prior.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

n, ukuran = 15, 28
gambar = np.zeros((ukuran * n, ukuran * n))
kisi_x = norm.ppf(np.linspace(0.05, 0.95, n))
kisi_y = norm.ppf(np.linspace(0.05, 0.95, n))[::-1]
for i, yi in enumerate(kisi_y):
    for j, xi in enumerate(kisi_x):
        z = np.array([[xi, yi]], dtype="float32")
        hasil = keras.ops.convert_to_numpy(decoder(z, training=False))[0, :, :, 0]
        gambar[i * ukuran:(i + 1) * ukuran, j * ukuran:(j + 1) * ukuran] = hasil

plt.figure(figsize=(10, 10)); plt.imshow(gambar, cmap="Greys_r"); plt.axis("off"); plt.show()

In [ ]:
(_, _), (x_uji, y_uji) = keras.datasets.mnist.load_data()
x_uji = np.expand_dims(x_uji, -1).astype("float32") / 255
z_mean_uji, _, _ = encoder.predict(x_uji, batch_size=256, verbose=0)
plt.figure(figsize=(8, 7))
plt.scatter(z_mean_uji[:, 0], z_mean_uji[:, 1], c=y_uji, cmap="tab10", s=2)
plt.colorbar(label="Angka"); plt.xlabel("z[0]"); plt.ylabel("z[1]")
plt.title("Sebaran data uji di ruang laten"); plt.show()